keyboard_arrow_down
🐋 Intelligent Whale & Dolphin Conservation — Colab demo

Capstone ASAC-CP-141 · Listen → Recognise → Predict → Warn

This notebook runs the complete flow on Google's free GPU, so nothing gets installed on your laptop.

How to use: Runtime → Change runtime type → T4 GPU, then Runtime → Run all (about 5–8 minutes).

Step	What happens
1	Audio → 5 s windows → log-mel spectrograms
2	Species classifier: baseline + Custom CNN + ResNet-50 + EfficientNet-B0 (transfer learning)
3	Noise-robustness test + Grad-CAM (explainable AI)
4	7-day whale-presence forecast
5	AIS ship traffic → ship-strike risk map + 10-knot what-if
6	Plain-language advisories for ships
7	Live demo: pick or upload a recording → species + Grad-CAM

⚠️ This uses synthetic sample data generated by our code, to test the full pipeline. Real Watkins / NOAA SanctSound / AIS data replaces it later (Gantt weeks 3–4).


keyboard_arrow_down
0 · Setup (GPU check + packages)


In [ ]:
import torch, subprocess, sys
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none - set Runtime > Change runtime type > T4 GPU")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "h3", "folium", "lightgbm", "soundfile"], check=False)
import os; os.makedirs("/content/whale_ai/src", exist_ok=True); os.chdir("/content/whale_ai")
print("ready:", os.getcwd())

GPU: Tesla T4
ready: /content/whale_ai



keyboard_arrow_down
Project code

Each cell below saves one file of our project (same code as in VS Code / GitHub). You can collapse this section.


In [ ]:
%%writefile src/config.py
"""Central settings for the whole project. Change values here, not inside the scripts."""
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
DATA = ROOT / "data"
RAW_CLIPS = DATA / "raw_clips"          # data/raw_clips/<species_name>/*.wav|*.flac  (Watkins clips go here)
NOISE_DIR = DATA / "noise"              # ship-noise-only clips used for SNR mixing
DETECTIONS_CSV = DATA / "detections" / "daily_presence.csv"   # columns: date, site, species, present (0/1)
AIS_DIR = DATA / "ais"                  # MarineCadastre AIS CSV files
SITES_CSV = DATA / "detections" / "sites.csv"                 # columns: site, lat, lon
SILVER = DATA / "silver"                # spectrogram arrays + metadata
OUTPUTS = ROOT / "outputs"              # figures, tables, models, reports

# ---- audio ----
SAMPLE_RATE = 16000        # Hz. NOAA files in our bucket are 16 kHz; everything is resampled to this
CLIP_SECONDS = 5.0         # window length fed to the model
HOP_SECONDS = 2.5          # 50% overlap between windows
N_FFT = 1024
HOP_LENGTH = 256
N_MELS = 128
FMIN, FMAX = 10, 8000      # blue/fin whales call at 10-100 Hz, dolphins whistle at several kHz

# ---- training ----
SEED = 42
TEST_SIZE = 0.15
VAL_SIZE = 0.15
SNR_LEVELS_DB = [20, 10, 5, 0]

# ---- risk engine ----
STUDY_BBOX = dict(lat_min=32.5, lat_max=35.0, lon_min=-121.5, lon_max=-117.0)  # Southern California Bight
GRID_DEG = 0.1             # fallback grid size when the h3 package is not installed (~11 km)
H3_RES = 6                 # ~36 km^2 hexagons when h3 is installed
LARGE_VESSEL_TYPES = set(range(60, 90))   # AIS codes: 60-69 passenger, 70-79 cargo, 80-89 tanker
SLOW_ZONE_KNOTS = 10.0

# Logistic curve fitted to Vanderlaan & Taggart (2007):
# P(lethal)=0.21 at 8.6 kn, 0.50 at 11.8 kn, 0.79 at 15 kn
LETHAL_B0, LETHAL_B1 = -4.89, 0.41

# ---- experiment tracking ----
MLFLOW_EXPERIMENT = "/Shared/whale-dolphin-capstone"   # Databricks path; local runs use ./mlruns

Writing src/config.py



In [ ]:
%%writefile src/audio_utils.py
"""Step 1-4 of the flow: load audio -> resample -> cut into windows -> log-mel spectrogram.
Only numpy + scipy are needed, so this runs anywhere (Mac, Databricks, Colab)."""
import numpy as np
from scipy import signal
from scipy.io import wavfile

from config import SAMPLE_RATE, CLIP_SECONDS, HOP_SECONDS, N_FFT, HOP_LENGTH, N_MELS, FMIN, FMAX


def load_audio(path, sr=SAMPLE_RATE):
    """Read WAV or FLAC, convert to mono float32 in [-1, 1], resample to `sr`."""
    path = str(path)
    try:
        import soundfile as sf          # needed for .flac (pip install soundfile)
        y, file_sr = sf.read(path, dtype="float32", always_2d=False)
    except ImportError:
        if path.lower().endswith(".flac"):
            raise ImportError("Reading FLAC needs the 'soundfile' package: pip install soundfile")
        file_sr, y = wavfile.read(path)
        if y.dtype.kind == "i":
            y = y.astype(np.float32) / np.iinfo(y.dtype).max
        y = y.astype(np.float32)
    if y.ndim > 1:
        y = y.mean(axis=1)
    if file_sr != sr:
        g = np.gcd(int(file_sr), int(sr))
        y = signal.resample_poly(y, sr // g, int(file_sr) // g).astype(np.float32)
    return y


def segment(y, sr=SAMPLE_RATE, clip_s=CLIP_SECONDS, hop_s=HOP_SECONDS, pad_short=True):
    """Cut audio into fixed windows (default 5 s with 50% overlap). Short clips are zero-padded."""
    n, h = int(clip_s * sr), int(hop_s * sr)
    if len(y) < n:
        return [np.pad(y, (0, n - len(y)))] if pad_short and len(y) > 0.5 * sr else []
    return [y[i:i + n] for i in range(0, len(y) - n + 1, h)]


def bandpass(y, sr=SAMPLE_RATE, low=FMIN, high=FMAX - 100):
    sos = signal.butter(4, [max(low, 5), min(high, sr / 2 - 50)], btype="band", fs=sr, output="sos")
    return signal.sosfiltfilt(sos, y).astype(np.float32)


def normalise(y):
    peak = np.max(np.abs(y)) + 1e-9
    return (y / peak).astype(np.float32)


_MEL_CACHE = {}


def mel_filterbank(sr=SAMPLE_RATE, n_fft=N_FFT, n_mels=N_MELS, fmin=FMIN, fmax=FMAX):
    key = (sr, n_fft, n_mels, fmin, fmax)
    if key in _MEL_CACHE:
        return _MEL_CACHE[key]
    hz2mel = lambda f: 2595 * np.log10(1 + f / 700)
    mel2hz = lambda m: 700 * (10 ** (m / 2595) - 1)
    mels = np.linspace(hz2mel(fmin), hz2mel(fmax), n_mels + 2)
    hz = mel2hz(mels)
    bins = np.fft.rfftfreq(n_fft, 1 / sr)
    fb = np.zeros((n_mels, len(bins)), dtype=np.float32)
    for m in range(1, n_mels + 1):
        l, c, r = hz[m - 1], hz[m], hz[m + 1]
        up = (bins - l) / (c - l + 1e-9)
        down = (r - bins) / (r - c + 1e-9)
        fb[m - 1] = np.maximum(0, np.minimum(up, down))
    _MEL_CACHE[key] = fb
    return fb


def log_mel(y, sr=SAMPLE_RATE):
    """Return a (N_MELS, frames) log-mel spectrogram in dB, scaled to roughly [0, 1]."""
    _, _, Z = signal.stft(y, fs=sr, nperseg=N_FFT, noverlap=N_FFT - HOP_LENGTH, boundary=None)
    power = np.abs(Z) ** 2
    mel = mel_filterbank(sr) @ power
    db = 10 * np.log10(mel + 1e-10)
    db = np.clip(db, db.max() - 80, None)          # keep an 80 dB dynamic range
    return ((db - db.min()) / (db.max() - db.min() + 1e-9)).astype(np.float32)


def mix_at_snr(clean, noise, snr_db, rng=None):
    """Add ship noise to a clip so that signal-to-noise ratio = snr_db (used for robustness tests)."""
    rng = rng or np.random.default_rng()
    if len(noise) < len(clean):
        noise = np.tile(noise, int(np.ceil(len(clean) / len(noise))))
    start = rng.integers(0, len(noise) - len(clean) + 1)
    noise = noise[start:start + len(clean)]
    p_sig = np.mean(clean ** 2) + 1e-12
    p_noise = np.mean(noise ** 2) + 1e-12
    scale = np.sqrt(p_sig / (p_noise * 10 ** (snr_db / 10)))
    return normalise(clean + scale * noise)


def spec_augment(spec, rng=None, n_freq_masks=2, n_time_masks=2, max_f=12, max_t=20):
    """SpecAugment: hide random frequency bands and time strips so the model cannot memorise."""
    rng = rng or np.random.default_rng()
    s = spec.copy()
    F, T = s.shape
    for _ in range(n_freq_masks):
        f = rng.integers(0, max_f); f0 = rng.integers(0, max(1, F - f)); s[f0:f0 + f, :] = 0
    for _ in range(n_time_masks):
        t = rng.integers(0, max_t); t0 = rng.integers(0, max(1, T - t)); s[:, t0:t0 + t] = 0
    return s

Writing src/audio_utils.py



In [ ]:
%%writefile src/tracking.py
"""Experiment tracking. Uses MLflow when it is installed (logs to Databricks if
MLFLOW_TRACKING_URI=databricks), otherwise saves the same info as JSON in outputs/runs/.

Connect your laptop to Databricks MLflow (one time):
    pip install mlflow databricks-sdk
    databricks auth login --host https://<your-workspace>.cloud.databricks.com
    export MLFLOW_TRACKING_URI=databricks
"""
import json
import os
import time
from pathlib import Path

from config import OUTPUTS, MLFLOW_EXPERIMENT

try:
    import mlflow
except ImportError:
    mlflow = None


class Run:
    def __init__(self, name, tags=None):
        self.name = name
        self.tags = tags or {}
        self.dir = OUTPUTS / "runs" / f"{time.strftime('%Y%m%d-%H%M%S')}_{name}"
        self.params, self.metrics = {}, {}
        self.use_mlflow = mlflow is not None

    def __enter__(self):
        self.dir.mkdir(parents=True, exist_ok=True)
        if self.use_mlflow:
            uri = os.environ.get("MLFLOW_TRACKING_URI", "")
            mlflow.set_experiment(MLFLOW_EXPERIMENT if uri.startswith("databricks") else "whale-dolphin-capstone")
            mlflow.start_run(run_name=self.name, tags=self.tags)
        return self

    def log_params(self, d):
        self.params.update(d)
        if self.use_mlflow:
            mlflow.log_params({k: str(v)[:250] for k, v in d.items()})

    def log_metrics(self, d, step=None):
        for k, v in d.items():
            self.metrics.setdefault(k, []).append((step, float(v)))
        if self.use_mlflow:
            mlflow.log_metrics({k: float(v) for k, v in d.items()}, step=step)

    def log_artifact(self, path):
        if self.use_mlflow:
            mlflow.log_artifact(str(path))

    def __exit__(self, *exc):
        last = {k: v[-1][1] for k, v in self.metrics.items()}
        (self.dir / "run.json").write_text(json.dumps(
            {"name": self.name, "tags": self.tags, "params": self.params, "final_metrics": last,
             "history": self.metrics}, indent=2, default=str))
        if self.use_mlflow:
            mlflow.end_run()
        print(f"[tracking] {self.name}: {json.dumps({k: round(v, 4) for k, v in last.items()})}")
        return False

Writing src/tracking.py



In [ ]:
%%writefile src/build_dataset.py
"""Bronze -> Silver: turn every audio file into 5-second log-mel spectrograms.

Input : data/raw_clips/<species>/*.wav|*.flac
Output: data/silver/specs.npy      (N, 128, frames) float16
        data/silver/meta.csv       file, species, start_s, split, noisy_copy

The train/val/test split is done BY FILE (not by window) so overlapping windows from the
same recording never appear in both train and test (avoids data leakage).
Use --noise-aug to add extra noisy copies of the TRAIN windows (objective O3).
"""
import argparse
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from config import RAW_CLIPS, NOISE_DIR, SILVER, SEED, TEST_SIZE, VAL_SIZE, SAMPLE_RATE, HOP_SECONDS
import audio_utils as au

AUDIO_EXT = {".wav", ".flac"}


def list_files():
    rows = []
    for d in sorted(p for p in RAW_CLIPS.iterdir() if p.is_dir()):
        for f in sorted(d.iterdir()):
            if f.suffix.lower() in AUDIO_EXT:
                rows.append((str(f), d.name))
    if not rows:
        raise SystemExit(f"No audio found in {RAW_CLIPS}. Run make_sample_data.py or add real clips.")
    return pd.DataFrame(rows, columns=["file", "species"])


def split_files(files, min_per_class=8):
    counts = files.species.value_counts()
    keep = counts[counts >= min_per_class].index
    dropped = sorted(set(files.species) - set(keep))
    if dropped:
        print("Skipping species with too few files:", dropped)
    files = files[files.species.isin(keep)].reset_index(drop=True)
    tr, te = train_test_split(files, test_size=TEST_SIZE, stratify=files.species, random_state=SEED)
    tr, va = train_test_split(tr, test_size=VAL_SIZE / (1 - TEST_SIZE), stratify=tr.species, random_state=SEED)
    files["split"] = "train"
    files.loc[va.index, "split"] = "val"
    files.loc[te.index, "split"] = "test"
    return files


def load_noise():
    if not NOISE_DIR.exists():
        return []
    return [au.load_audio(f) for f in sorted(NOISE_DIR.iterdir()) if f.suffix.lower() in AUDIO_EXT]


def window_audio(path):
    y = au.normalise(au.bandpass(au.load_audio(path)))
    return au.segment(y)


def main(noise_aug, max_windows_per_file):
    rng = np.random.default_rng(SEED)
    files = split_files(list_files())
    noises = load_noise() if noise_aug else []
    specs, meta = [], []
    for i, r in files.iterrows():
        try:
            wins = window_audio(r.file)[:max_windows_per_file]
        except Exception as e:                       # corrupted file -> skip, but tell the user
            print("  skip", r.file, e)
            continue
        for k, w in enumerate(wins):
            specs.append(au.log_mel(w).astype(np.float16))
            meta.append((r.file, r.species, k * HOP_SECONDS, r.split, 0))
            if noises and r.split == "train" and r.species != "background":
                noisy = au.mix_at_snr(w, noises[rng.integers(len(noises))], rng.uniform(0, 20), rng)
                specs.append(au.log_mel(noisy).astype(np.float16))
                meta.append((r.file, r.species, k * HOP_SECONDS, r.split, 1))
        if (i + 1) % 100 == 0:
            print(f"  {i + 1}/{len(files)} files")
    SILVER.mkdir(parents=True, exist_ok=True)
    X = np.stack(specs)
    np.save(SILVER / "specs.npy", X)
    meta = pd.DataFrame(meta, columns=["file", "species", "start_s", "split", "noisy_copy"])
    meta.to_csv(SILVER / "meta.csv", index=False)
    print(f"Saved {X.shape} spectrograms to {SILVER}")
    print(meta.groupby(["split", "species"]).size().unstack(0).fillna(0).astype(int))


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--noise-aug", action="store_true", help="add noisy copies of train windows (O3)")
    ap.add_argument("--max-windows-per-file", type=int, default=6)
    a = ap.parse_args()
    main(a.noise_aug, a.max_windows_per_file)

Writing src/build_dataset.py



In [ ]:
%%writefile src/evaluate.py
"""Shared evaluation helpers: load silver data, metrics, confusion matrix, noise-robustness test."""
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix, roc_auc_score

from config import SILVER, OUTPUTS, NOISE_DIR, SNR_LEVELS_DB, SEED
import audio_utils as au


def load_silver(include_noisy_copies=True):
    X = np.load(SILVER / "specs.npy").astype(np.float32)
    meta = pd.read_csv(SILVER / "meta.csv")
    if not include_noisy_copies:
        keep = meta.noisy_copy.values == 0
        X, meta = X[keep], meta[keep].reset_index(drop=True)
    classes = sorted(meta.species.unique())
    y = meta.species.map({c: i for i, c in enumerate(classes)}).values
    return X, y, meta, classes


def metrics(y_true, proba, classes):
    pred = proba.argmax(1)
    out = {"accuracy": accuracy_score(y_true, pred),
           "macro_f1": f1_score(y_true, pred, average="macro")}
    try:
        out["macro_auc"] = roc_auc_score(y_true, proba, multi_class="ovr", labels=list(range(len(classes))))
    except ValueError:
        pass
    return out


def save_report(y_true, proba, classes, name):
    d = OUTPUTS / "figures"; d.mkdir(parents=True, exist_ok=True)
    pred = proba.argmax(1)
    rep = classification_report(y_true, pred, labels=list(range(len(classes))), target_names=classes,
                                digits=3, zero_division=0, output_dict=True)
    (OUTPUTS / "reports").mkdir(parents=True, exist_ok=True)
    (OUTPUTS / "reports" / f"{name}_classification_report.json").write_text(json.dumps(rep, indent=2))
    cm = confusion_matrix(y_true, pred, labels=list(range(len(classes))))
    cmn = cm / cm.sum(1, keepdims=True).clip(min=1)
    fig, ax = plt.subplots(figsize=(6.2, 5.2))
    ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
    short = [c.replace("_", "\n") for c in classes]
    ax.set_xticks(range(len(classes)), short, fontsize=8); ax.set_yticks(range(len(classes)), short, fontsize=8)
    for i in range(len(classes)):
        for j in range(len(classes)):
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=9,
                    color="white" if cmn[i, j] > 0.6 else "#1f232b")
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(f"Confusion matrix — {name}")
    fig.tight_layout()
    path = d / f"{name}_confusion.png"
    fig.savefig(path, dpi=160); plt.close(fig)
    return path


def noise_robustness(predict_fn, meta, classes, name, max_windows=400):
    """Re-load clean TEST windows, add ship noise at each SNR, measure macro-F1.
    predict_fn: list[spectrogram] -> probabilities (N, C)."""
    noises = [au.load_audio(f) for f in sorted(NOISE_DIR.glob("*")) if f.suffix.lower() in {".wav", ".flac"}]
    if not noises:
        print("No noise clips in data/noise — skipping noise test"); return None
    rng = np.random.default_rng(SEED)
    test = meta[(meta.split == "test") & (meta.noisy_copy == 0) & (meta.species != "background")]
    test = test.sample(min(max_windows, len(test)), random_state=SEED)
    cache, idx = {}, {c: i for i, c in enumerate(classes)}
    clean_wins, labels = [], []
    for r in test.itertuples():
        if r.file not in cache:
            y = au.normalise(au.bandpass(au.load_audio(r.file)))
            cache[r.file] = au.segment(y)
        k = int(round(r.start_s / 2.5))
        if k < len(cache[r.file]):
            clean_wins.append(cache[r.file][k]); labels.append(idx[r.species])
    labels = np.array(labels)
    results = {}
    for snr in ["clean"] + SNR_LEVELS_DB:
        specs = []
        for w in clean_wins:
            ww = w if snr == "clean" else au.mix_at_snr(w, noises[rng.integers(len(noises))], snr, rng)
            specs.append(au.log_mel(ww))
        proba = predict_fn(specs)
        results[str(snr)] = f1_score(labels, proba.argmax(1), average="macro")
    d = OUTPUTS / "figures"; d.mkdir(parents=True, exist_ok=True)
    fig, ax = plt.subplots(figsize=(5.5, 3.6))
    xs = list(results.keys())
    ax.plot(xs, list(results.values()), marker="o", color="#8E0A13", lw=2)
    ax.set_ylim(0, 1.02); ax.set_ylabel("Macro-F1"); ax.set_xlabel("Signal-to-noise ratio (dB)")
    ax.set_title(f"Noise robustness — {name}"); ax.grid(alpha=.3)
    fig.tight_layout(); fig.savefig(d / f"{name}_noise_curve.png", dpi=160); plt.close(fig)
    return results

Writing src/evaluate.py



In [ ]:
%%writefile src/train_baseline.py
"""Classical baseline: mel-band statistics + Logistic Regression.
Fast (seconds, CPU only). Every deep model in train_cnn.py must beat this."""
import joblib
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from config import OUTPUTS
from evaluate import load_silver, metrics, save_report, noise_robustness
from tracking import Run


def features(specs):
    """Per mel band: mean, std and mean absolute time-difference -> 3 x 128 numbers per clip."""
    S = np.asarray(specs, dtype=np.float32)
    return np.concatenate([S.mean(2), S.std(2), np.abs(np.diff(S, axis=2)).mean(2)], axis=1)


def main(name="baseline_logreg"):
    X, y, meta, classes = load_silver()
    tr = (meta.split == "train").values
    te = (meta.split == "test").values
    F = features(X)
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, C=0.5, class_weight="balanced"))
    with Run(name, tags={"module": "A", "family": "classical"}) as run:
        run.log_params({"model": "LogisticRegression", "features": "mel mean/std/delta (384)",
                        "n_train": int(tr.sum()), "n_test": int(te.sum()), "classes": ",".join(classes)})
        model.fit(F[tr], y[tr])
        proba = model.predict_proba(F[te])
        run.log_metrics(metrics(y[te], proba, classes))
        run.log_artifact(save_report(y[te], proba, classes, name))
        res = noise_robustness(lambda specs: model.predict_proba(features(specs)), meta, classes, name)
        if res:
            run.log_metrics({f"f1_snr_{k}": v for k, v in res.items()})
            run.log_artifact(OUTPUTS / "figures" / f"{name}_noise_curve.png")
        (OUTPUTS / "models").mkdir(parents=True, exist_ok=True)
        joblib.dump({"model": model, "classes": classes}, OUTPUTS / "models" / f"{name}.joblib")
    return model, classes


if __name__ == "__main__":
    main()

Writing src/train_baseline.py



In [ ]:
%%writefile src/train_cnn.py
"""Module A deep models: custom CNN vs ImageNet transfer learning (ResNet-50, EfficientNet-B0).
The spectrogram is treated like an image (same idea as chest X-ray transfer learning).

    python src/train_cnn.py --model custom_cnn
    python src/train_cnn.py --model resnet50 --epochs 15
    python src/train_cnn.py --model efficientnet_b0 --epochs 15

Runs on Apple Silicon GPU (MPS) automatically. Every epoch is logged to MLflow.
"""
import argparse
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from config import OUTPUTS, SEED
from evaluate import load_silver, metrics, save_report, noise_robustness
from tracking import Run
import audio_utils as au

IMAGENET_MEAN, IMAGENET_STD = 0.449, 0.226     # grey-scale version of ImageNet normalisation


def device():
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


class SpecDataset(Dataset):
    def __init__(self, X, y, augment=False):
        self.X, self.y, self.augment = X, y, augment
        self.rng = np.random.default_rng(SEED)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        s = self.X[i]
        if self.augment:
            s = au.spec_augment(s, self.rng)
        return torch.from_numpy(np.ascontiguousarray(s))[None], int(self.y[i])


class CustomCNN(nn.Module):
    """4 conv blocks — the 'from scratch' baseline."""
    def __init__(self, n_classes):
        super().__init__()
        def block(i, o):
            return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(), nn.MaxPool2d(2))
        self.features = nn.Sequential(block(1, 16), block(16, 32), block(32, 64), block(64, 128))
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(128, n_classes))

    def forward(self, x):
        return self.head(self.features(x))


class ImageNetWrapper(nn.Module):
    """Repeats the 1-channel spectrogram to 3 channels and normalises like ImageNet."""
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone

    def forward(self, x):
        x = (x - IMAGENET_MEAN) / IMAGENET_STD
        return self.backbone(x.repeat(1, 3, 1, 1))


def build_model(name, n_classes):
    from torchvision import models
    if name == "custom_cnn":
        return CustomCNN(n_classes)
    if name == "resnet50":
        m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        m.fc = nn.Linear(m.fc.in_features, n_classes)
        return ImageNetWrapper(m)
    if name == "efficientnet_b0":
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, n_classes)
        return ImageNetWrapper(m)
    raise ValueError(name)


def last_conv_layer(model):
    """Layer used by Grad-CAM."""
    if isinstance(model, CustomCNN):
        return model.features[-1][0]
    bb = model.backbone
    return bb.layer4[-1] if hasattr(bb, "layer4") else bb.features[-1]


@torch.no_grad()
def predict(model, specs, dev, bs=64):
    model.eval()
    out = []
    for i in range(0, len(specs), bs):
        x = torch.from_numpy(np.asarray(specs[i:i + bs], dtype=np.float32))[:, None].to(dev)
        out.append(torch.softmax(model(x), 1).cpu().numpy())
    return np.concatenate(out)


def main(name, epochs, lr, bs, patience, no_noisy_copies):
    torch.manual_seed(SEED)
    X, y, meta, classes = load_silver(include_noisy_copies=not no_noisy_copies)
    split = meta.split.values
    dev = device()
    model = build_model(name, len(classes)).to(dev)
    counts = np.bincount(y[split == "train"], minlength=len(classes))
    weights = torch.tensor(len(y) / (len(classes) * np.maximum(counts, 1)), dtype=torch.float32, device=dev)
    loss_fn = nn.CrossEntropyLoss(weight=weights)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    train_dl = DataLoader(SpecDataset(X[split == "train"], y[split == "train"], augment=True), batch_size=bs, shuffle=True)
    run_name = name + ("_clean" if no_noisy_copies else "")
    ckpt = OUTPUTS / "models" / f"{run_name}.pt"
    ckpt.parent.mkdir(parents=True, exist_ok=True)
    best, bad = -1, 0
    with Run(run_name, tags={"module": "A", "family": "deep"}) as run:
        run.log_params({"model": name, "epochs": epochs, "lr": lr, "batch_size": bs, "device": str(dev),
                        "noise_augmented_training": not no_noisy_copies, "classes": ",".join(classes)})
        for ep in range(1, epochs + 1):
            model.train(); total = 0
            for xb, yb in train_dl:
                xb, yb = xb.to(dev), yb.to(dev)
                opt.zero_grad(); loss = loss_fn(model(xb), yb); loss.backward(); opt.step()
                total += loss.item() * len(yb)
            sched.step()
            val = metrics(y[split == "val"], predict(model, X[split == "val"], dev), classes)
            run.log_metrics({"train_loss": total / (split == "train").sum(), "val_macro_f1": val["macro_f1"]}, step=ep)
            print(f"epoch {ep:02d}  loss {total / (split == 'train').sum():.3f}  val macro-F1 {val['macro_f1']:.3f}")
            if val["macro_f1"] > best:
                best, bad = val["macro_f1"], 0
                torch.save({"state": model.state_dict(), "classes": classes, "name": name}, ckpt)
            else:
                bad += 1
                if bad >= patience:
                    print("early stopping"); break
        model.load_state_dict(torch.load(ckpt, map_location=dev)["state"])
        te = split == "test"
        proba = predict(model, X[te], dev)
        run.log_metrics({f"test_{k}": v for k, v in metrics(y[te], proba, classes).items()})
        run.log_artifact(save_report(y[te], proba, classes, run_name))
        res = noise_robustness(lambda s: predict(model, s, dev), meta, classes, run_name)
        if res:
            run.log_metrics({f"f1_snr_{k}": v for k, v in res.items()})
        run.log_artifact(ckpt)
        try:                                     # Grad-CAM figure for the report (objective O4)
            from gradcam import gradcam_figure
            run.log_artifact(gradcam_figure(model, X[te], y[te], classes, run_name, dev, last_conv_layer(model)))
        except Exception as e:
            print("Grad-CAM skipped:", e)


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", default="custom_cnn", choices=["custom_cnn", "resnet50", "efficientnet_b0"])
    ap.add_argument("--epochs", type=int, default=20)
    ap.add_argument("--lr", type=float, default=3e-4)
    ap.add_argument("--batch-size", type=int, default=32)
    ap.add_argument("--patience", type=int, default=5)
    ap.add_argument("--no-noisy-copies", action="store_true", help="train only on clean windows (for O3 comparison)")
    a = ap.parse_args()
    main(a.model, a.epochs, a.lr, a.batch_size, a.patience, a.no_noisy_copies)

Writing src/train_cnn.py



In [ ]:
%%writefile src/gradcam.py
"""Objective O4 — Grad-CAM on spectrograms: which time-frequency region made the model decide?"""
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from config import OUTPUTS, CLIP_SECONDS, FMIN, FMAX


def gradcam(model, spec, target_class, layer, dev):
    acts, grads = {}, {}
    def fwd_hook(m, i, o):                  # tensor hook avoids problems with in-place ReLU in ResNet
        acts["a"] = o
        o.register_hook(lambda g: grads.__setitem__("g", g))
    h1 = layer.register_forward_hook(fwd_hook)
    model.eval()
    x = torch.from_numpy(np.asarray(spec, dtype=np.float32))[None, None].to(dev).requires_grad_(True)
    out = model(x)
    model.zero_grad(); out[0, target_class].backward()
    h1.remove()
    a, g = acts["a"][0], grads["g"][0]                  # (C, h, w)
    cam = F.relu((g.mean((1, 2), keepdim=True) * a).sum(0))
    cam = F.interpolate(cam[None, None], size=spec.shape, mode="bilinear", align_corners=False)[0, 0]
    cam = cam.detach().cpu().numpy()
    return (cam - cam.min()) / (cam.max() - cam.min() + 1e-9), torch.softmax(out, 1)[0].detach().cpu().numpy()


def gradcam_figure(model, X, y, classes, name, dev, layer):
    picks = [np.where(y == c)[0][0] for c in range(len(classes)) if (y == c).any()]
    fig, axes = plt.subplots(2, len(picks), figsize=(2.6 * len(picks), 4.6))
    axes = np.atleast_2d(axes)
    for j, i in enumerate(picks):
        cam, p = gradcam(model, X[i], int(y[i]), layer, dev)
        ext = [0, CLIP_SECONDS, 0, X[i].shape[0]]
        axes[0, j].imshow(X[i], origin="lower", aspect="auto", cmap="magma", extent=ext)
        axes[0, j].set_title(classes[y[i]].replace("_", " "), fontsize=9)
        axes[1, j].imshow(X[i], origin="lower", aspect="auto", cmap="gray", extent=ext)
        axes[1, j].imshow(cam, origin="lower", aspect="auto", cmap="jet", alpha=0.45, extent=ext)
        axes[1, j].set_title(f"pred: {classes[p.argmax()].replace('_', ' ')} ({p.max():.2f})", fontsize=8)
        for a in axes[:, j]:
            a.set_xticks([]); a.set_yticks([])
    axes[0, 0].set_ylabel("log-mel"); axes[1, 0].set_ylabel("Grad-CAM")
    fig.suptitle(f"Grad-CAM — {name} (mel {FMIN}-{FMAX} Hz, {CLIP_SECONDS:.0f} s)", fontsize=10)
    fig.tight_layout()
    path = OUTPUTS / "figures" / f"{name}_gradcam.png"
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=150); plt.close(fig)
    return path

Writing src/gradcam.py



In [ ]:
%%writefile src/forecast.py
"""Module B1 — Will whales be present at a site in the NEXT 7 days?

Input : data/detections/daily_presence.csv  (date, site, species, present)
        Real data: NOAA SanctSound daily presence products (ERDDAP), see README.
Models: seasonal baseline | logistic regression | gradient boosting (LightGBM if installed)
Split : by time — train on earlier years, test on the last year (no peeking into the future).
Output: outputs/tables/forecast_latest.csv  (site, species, p_next7)
"""
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss, f1_score

from config import DETECTIONS_CSV, OUTPUTS
from tracking import Run

try:
    from lightgbm import LGBMClassifier as GBM
    GBM_KW = dict(n_estimators=300, learning_rate=0.05, num_leaves=31, verbose=-1)
except Exception:          # not installed, or libomp missing on Mac (brew install libomp)
    from sklearn.ensemble import HistGradientBoostingClassifier as GBM
    GBM_KW = dict(max_iter=300, learning_rate=0.05)

FEATURES = ["doy_sin", "doy_cos", "rate_7", "rate_14", "rate_30", "present_today", "site_code", "species_code"]


def make_features(df):
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values(["site", "species", "date"])
    g = df.groupby(["site", "species"])["present"]
    for w in (7, 14, 30):
        df[f"rate_{w}"] = g.transform(lambda s: s.rolling(w, min_periods=1).mean())
    df["present_today"] = df["present"]
    # target: any presence in the next 7 days (t+1 ... t+7)
    df["target"] = g.transform(lambda s: s[::-1].rolling(7, min_periods=7).max()[::-1].shift(-1))
    doy = df.date.dt.dayofyear
    df["doy_sin"], df["doy_cos"] = np.sin(2 * np.pi * doy / 365.25), np.cos(2 * np.pi * doy / 365.25)
    df["site_code"] = df.site.astype("category").cat.codes
    df["species_code"] = df.species.astype("category").cat.codes
    df["week"] = df.date.dt.isocalendar().week.astype(int)
    return df


def seasonal_baseline(train, test):
    """Probability = how often whales were present in the same week of year in past years."""
    rate = train.groupby(["site", "species", "week"])["target"].mean().rename("p")
    return test.join(rate, on=["site", "species", "week"])["p"].fillna(train.target.mean()).values


def main():
    df = make_features(pd.read_csv(DETECTIONS_CSV)).dropna(subset=["target"])
    last_year = df.date.dt.year.max()
    train, test = df[df.date.dt.year < last_year], df[df.date.dt.year == last_year]
    results, probs = {}, {}
    probs["seasonal_baseline"] = seasonal_baseline(train, test)
    lr = LogisticRegression(max_iter=2000).fit(train[FEATURES], train.target)
    probs["logistic_regression"] = lr.predict_proba(test[FEATURES])[:, 1]
    gbm = GBM(**GBM_KW).fit(train[FEATURES], train.target)
    probs["gradient_boosting"] = gbm.predict_proba(test[FEATURES])[:, 1]
    for name, p in probs.items():
        with Run(f"forecast_{name}", tags={"module": "B1"}) as run:
            run.log_params({"train_years": f"{train.date.dt.year.min()}-{last_year - 1}", "test_year": last_year,
                            "n_train": len(train), "n_test": len(test), "gbm_impl": GBM.__name__})
            m = {"auc": roc_auc_score(test.target, p), "brier": brier_score_loss(test.target, p),
                 "f1": f1_score(test.target, p >= 0.5)}
            run.log_metrics(m)
            results[name] = m
    # plot one site/species to show the forecast vs reality
    d = OUTPUTS / "figures"; d.mkdir(parents=True, exist_ok=True)
    sel = (test.site == test.site.iloc[0]) & (test.species == "blue_whale" if "blue_whale" in set(test.species) else test.species == test.species.iloc[0])
    t = test[sel]
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.fill_between(t.date, 0, t.target, step="mid", color="#D5D8DE", label="actual presence (next 7 days)")
    ax.plot(t.date, probs["gradient_boosting"][sel.values], color="#8E0A13", lw=1.6, label="gradient boosting forecast")
    ax.plot(t.date, probs["seasonal_baseline"][sel.values], color="#012258", lw=1.2, ls="--", label="seasonal baseline")
    ax.set_ylim(0, 1.05); ax.set_ylabel("P(present)"); ax.legend(fontsize=8, loc="upper left")
    ax.set_title(f"7-day presence forecast — {t.site.iloc[0]}, {t.species.iloc[0].replace('_', ' ')}, {last_year}")
    fig.tight_layout(); fig.savefig(d / "forecast_example.png", dpi=160); plt.close(fig)
    # latest forecast for the risk engine: use the most recent day in the data
    full = make_features(pd.read_csv(DETECTIONS_CSV))
    latest = full[full.date == full.date.max()].copy()
    latest["p_next7"] = gbm.predict_proba(latest[FEATURES])[:, 1]
    (OUTPUTS / "tables").mkdir(parents=True, exist_ok=True)
    latest[["date", "site", "species", "p_next7"]].to_csv(OUTPUTS / "tables" / "forecast_latest.csv", index=False)
    pd.DataFrame(results).T.round(3).to_csv(OUTPUTS / "tables" / "forecast_metrics.csv")
    print(pd.DataFrame(results).T.round(3))
    return results


if __name__ == "__main__":
    main()

Writing src/forecast.py



In [ ]:
%%writefile src/risk_engine.py
"""Module B2 + B3 — Ship traffic from AIS and the dynamic ship-strike risk score.

Risk(cell, week) = P(whale) x ship-km x P(lethal | speed)

  P(whale)          from forecast.py (nearest acoustic site, decays with distance)
  ship-km           distance sailed by large vessels inside the cell (from AIS)
  P(lethal | speed) logistic curve fitted to Vanderlaan & Taggart (2007)

Also runs the "what if all ships slow to 10 knots?" scenario.
Input : data/ais/*.csv (MarineCadastre format, old or new column names), outputs/tables/forecast_latest.csv
Output: outputs/tables/risk_cells.csv, outputs/figures/risk_map.png (+ risk_map.html if folium installed)
"""
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from config import (AIS_DIR, SITES_CSV, OUTPUTS, STUDY_BBOX, GRID_DEG, H3_RES, LARGE_VESSEL_TYPES,
                    SLOW_ZONE_KNOTS, LETHAL_B0, LETHAL_B1)

try:
    import h3
except ImportError:
    h3 = None

COLS = {"mmsi": "MMSI", "base_date_time": "BaseDateTime", "basedatetime": "BaseDateTime", "latitude": "LAT",
        "lat": "LAT", "longitude": "LON", "lon": "LON", "sog": "SOG", "vessel_type": "VesselType",
        "vesseltype": "VesselType"}


def p_lethal(knots):
    return 1 / (1 + np.exp(-(LETHAL_B0 + LETHAL_B1 * np.asarray(knots, dtype=float))))


def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(a))


def load_ais():
    files = sorted(AIS_DIR.glob("*.csv"))
    if not files:
        raise SystemExit(f"No AIS CSV in {AIS_DIR}")
    usecols = lambda c: c.lower() in COLS or c in COLS.values()
    df = pd.concat([pd.read_csv(f, usecols=usecols) for f in files], ignore_index=True)
    df = df.rename(columns={c: COLS.get(c.lower(), c) for c in df.columns})
    b = STUDY_BBOX
    df = df[df.LAT.between(b["lat_min"], b["lat_max"]) & df.LON.between(b["lon_min"], b["lon_max"])]
    df = df[df.VesselType.isin(LARGE_VESSEL_TYPES) & (df.SOG >= 1) & (df.SOG < 40)]
    df["BaseDateTime"] = pd.to_datetime(df.BaseDateTime)
    return df.sort_values(["MMSI", "BaseDateTime"]).reset_index(drop=True)


def cell_id(lat, lon):
    if h3 is not None:
        fn = getattr(h3, "latlng_to_cell", None) or getattr(h3, "geo_to_h3")
        return [fn(a, o, H3_RES) for a, o in zip(lat, lon)]
    return [f"{np.floor(a / GRID_DEG) * GRID_DEG:.2f}_{np.floor(o / GRID_DEG) * GRID_DEG:.2f}" for a, o in zip(lat, lon)]


def cell_center(c):
    if h3 is not None:
        fn = getattr(h3, "cell_to_latlng", None) or getattr(h3, "h3_to_geo")
        return fn(c)
    a, o = map(float, c.split("_"))
    return a + GRID_DEG / 2, o + GRID_DEG / 2


def traffic_by_cell(ais):
    """Ship-km and km-weighted lethality per cell per week."""
    g = ais.groupby("MMSI")
    ais["km"] = haversine_km(g.LAT.shift(), g.LON.shift(), ais.LAT, ais.LON)
    gap_h = (ais.BaseDateTime - g.BaseDateTime.shift()).dt.total_seconds() / 3600
    ais.loc[(gap_h > 1) | ais.km.isna() | (ais.km > 50), "km"] = 0     # ignore jumps / AIS gaps
    ais["cell"] = cell_id(ais.LAT.values, ais.LON.values)
    ais["week"] = ais.BaseDateTime.dt.to_period("W").astype(str)
    ais["leth_km"] = ais.km * p_lethal(ais.SOG)
    ais["leth_km_slow"] = ais.km * p_lethal(np.minimum(ais.SOG, SLOW_ZONE_KNOTS))
    t = ais.groupby(["cell", "week"]).agg(ship_km=("km", "sum"), leth_km=("leth_km", "sum"),
                                          leth_km_slow=("leth_km_slow", "sum"), mean_sog=("SOG", "mean"),
                                          vessels=("MMSI", "nunique")).reset_index()
    return t[t.ship_km > 0]


def whale_probability(cells, forecast, sites, scale_km=40.0):
    """P(any whale species present) at each cell: nearest site's forecast, decaying with distance.
    This is a simple, clearly stated modelling assumption (to be refined with real detection ranges)."""
    f = forecast.groupby("site")["p_next7"].apply(lambda p: 1 - np.prod(1 - p)).rename("p_site")
    s = sites.join(f, on="site").dropna()
    centers = np.array([cell_center(c) for c in cells])
    d = haversine_km(centers[:, None, 0], centers[:, None, 1], s.lat.values[None], s.lon.values[None])
    contrib = s.p_site.values[None] * np.exp(-d / scale_km)
    return contrib.max(1), centers


def main():
    ais = load_ais()
    traffic = traffic_by_cell(ais)
    week_tbl = traffic.groupby("cell").agg(ship_km=("ship_km", "mean"), leth_km=("leth_km", "mean"),
                                           leth_km_slow=("leth_km_slow", "mean"), mean_sog=("mean_sog", "mean"),
                                           vessels=("vessels", "sum")).reset_index()     # average week
    forecast = pd.read_csv(OUTPUTS / "tables" / "forecast_latest.csv")
    sites = pd.read_csv(SITES_CSV)
    week_tbl["p_whale"], centers = whale_probability(week_tbl.cell, forecast, sites)
    week_tbl["lat"], week_tbl["lon"] = centers[:, 0], centers[:, 1]
    week_tbl["risk"] = week_tbl.p_whale * week_tbl.leth_km          # = P(whale) x ship-km x mean P(lethal)
    week_tbl["risk_if_10kn"] = week_tbl.p_whale * week_tbl.leth_km_slow
    q90, q70 = week_tbl.risk.quantile([0.9, 0.7])
    week_tbl["risk_level"] = np.where(week_tbl.risk >= q90, "HIGH", np.where(week_tbl.risk >= q70, "MEDIUM", "LOW"))
    week_tbl["reduction_pct"] = 100 * (1 - week_tbl.risk_if_10kn / week_tbl.risk.clip(lower=1e-12))
    week_tbl = week_tbl.sort_values("risk", ascending=False)
    (OUTPUTS / "tables").mkdir(parents=True, exist_ok=True)
    week_tbl.to_csv(OUTPUTS / "tables" / "risk_cells.csv", index=False)
    total_red = 100 * (1 - week_tbl.risk_if_10kn.sum() / week_tbl.risk.sum())
    high_red = week_tbl[week_tbl.risk_level == "HIGH"].reduction_pct.mean()
    plot(week_tbl, sites, total_red)
    summary = {"cells": len(week_tbl), "high_cells": int((week_tbl.risk_level == "HIGH").sum()),
               "risk_reduction_if_10kn_pct": round(total_red, 1), "high_cell_reduction_pct": round(high_red, 1),
               "ais_points": len(ais), "vessels": int(ais.MMSI.nunique()), "grid": "h3" if h3 else f"{GRID_DEG} deg"}
    print(summary)
    return summary


def plot(t, sites, total_red):
    colors = {"HIGH": "#8E0A13", "MEDIUM": "#E0A100", "LOW": "#9DB4D3"}
    fig, ax = plt.subplots(figsize=(8, 5))
    for lvl in ["LOW", "MEDIUM", "HIGH"]:
        s = t[t.risk_level == lvl]
        ax.scatter(s.lon, s.lat, s=40 + 300 * s.risk / t.risk.max(), c=colors[lvl], marker="s",
                   alpha=0.85, label=f"{lvl} ({len(s)})", edgecolors="none")
    ax.scatter(sites.lon, sites.lat, marker="^", s=90, c="#012258", label="acoustic sites")
    for r in sites.itertuples():
        ax.annotate(r.site, (r.lon, r.lat), textcoords="offset points", xytext=(5, 5), fontsize=8, color="#012258")
    ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude"); ax.grid(alpha=.25)
    ax.set_title(f"Weekly ship-strike risk by cell — slowing to {SLOW_ZONE_KNOTS:.0f} kn cuts total risk by {total_red:.0f}%")
    ax.legend(fontsize=8, loc="lower left")
    fig.tight_layout()
    p = OUTPUTS / "figures" / "risk_map.png"; p.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(p, dpi=160); plt.close(fig)
    try:
        import folium
        m = folium.Map(location=[33.9, -119.5], zoom_start=8, tiles="CartoDB positron")
        for r in t.itertuples():
            folium.CircleMarker([r.lat, r.lon], radius=4 + 10 * r.risk / t.risk.max(), color=colors[r.risk_level],
                                fill=True, fill_opacity=0.8,
                                popup=f"{r.risk_level} risk | P(whale) {r.p_whale:.2f} | {r.ship_km:.0f} ship-km/wk | "
                                      f"avg {r.mean_sog:.1f} kn | -{r.reduction_pct:.0f}% at 10 kn").add_to(m)
        m.save(str(OUTPUTS / "figures" / "risk_map.html"))
    except ImportError:
        pass


if __name__ == "__main__":
    main()

Writing src/risk_engine.py



In [ ]:
%%writefile src/advisory.py
"""Module B4 — Turn the risk table into plain-language advisories for ships.

On Databricks the text is written by an LLM with ai_query() (see databricks/03_ai_advisory.sql).
Locally (no Databricks), a rule-based template produces the same kind of message so the
flow can be demonstrated end to end.
"""
import pandas as pd

from config import OUTPUTS, SLOW_ZONE_KNOTS


def template_advisory(r):
    return (f"[{r.risk_level}] Zone {r.cell} (≈{r.lat:.2f}°N, {abs(r.lon):.2f}°W): whale presence likely this week "
            f"(P≈{r.p_whale:.2f}). Ships here average {r.mean_sog:.1f} kn. Reduce speed to {SLOW_ZONE_KNOTS:.0f} kn "
            f"or less — this cuts lethal-strike risk by ~{r.reduction_pct:.0f}%. Keep a lookout and report sightings.")


def main(top_n=5):
    t = pd.read_csv(OUTPUTS / "tables" / "risk_cells.csv")
    high = t[t.risk_level == "HIGH"].head(top_n)
    lines = [template_advisory(r) for r in high.itertuples()]
    out = OUTPUTS / "reports" / "advisories.txt"
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text("WHALE SHIP-STRIKE ADVISORIES (auto-generated)\n\n" + "\n\n".join(lines) + "\n")
    print("\n".join(lines[:2]))
    return lines


if __name__ == "__main__":
    main()

Writing src/advisory.py



In [ ]:
%%writefile src/run_pipeline.py
"""Runs the COMPLETE FLOW in order, end to end:

  1 data  ->  2 spectrograms  ->  3 species model  ->  4 noise test  ->  5 forecast
  ->  6 AIS + risk engine  ->  7 advisories  ->  8 summary report

    python src/run_pipeline.py --sample          # synthetic sample data (tests the code)
    python src/run_pipeline.py                   # your real data in data/
    python src/run_pipeline.py --sample --cnn custom_cnn   # also train a deep model (needs torch)
"""
import sys
import os
sys.path.insert(0, "/content/whale_ai/src")
sys.path.insert(0, os.getcwd())

import argparse
import json
import time
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from config import OUTPUTS, SILVER, CLIP_SECONDS


def step(n, title):
    print(f"\n{'=' * 70}\nSTEP {n}: {title}\n{'=' * 70}")
    return time.time()


def spectrogram_gallery():
    X = np.load(SILVER / "specs.npy", mmap_mode="r")
    meta = pd.read_csv(SILVER / "meta.csv")
    classes = sorted(meta.species.unique())
    fig, axes = plt.subplots(1, len(classes), figsize=(2.4 * len(classes), 2.8))
    for a, c in zip(np.atleast_1d(axes), classes):
        i = meta.index[(meta.species == c) & (meta.noisy_copy == 0)][0]
        a.imshow(np.asarray(X[i], dtype=np.float32), origin="lower", aspect="auto", cmap="magma",
                 extent=[0, CLIP_SECONDS, 0, X.shape[1]])
        a.set_title(c.replace("_", " "), fontsize=9); a.set_xlabel("time (s)", fontsize=8)
        a.set_yticks([0, 64, 127], ["10 Hz", "1.3 kHz", "8 kHz"], fontsize=7)
    fig.tight_layout()
    p = OUTPUTS / "figures" / "spectrogram_gallery.png"; p.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(p, dpi=160); plt.close(fig)


def main(sample, cnn, clips_per_class):
    t_all = time.time(); summary = {"data": "SYNTHETIC SAMPLE (code test only)" if sample else "real data"}
    if sample:
        t = step(1, "Create synthetic sample data (replace with real data later)")
        import make_sample_data; make_sample_data.main(clips_per_class)
    t = step(2, "Audio -> 5 s windows -> log-mel spectrograms (Bronze -> Silver)")
    import build_dataset; build_dataset.main(noise_aug=True, max_windows_per_file=6)
    spectrogram_gallery()
    t = step(3, "Species classifier: classical baseline (+ noise robustness test)")
    import train_baseline; train_baseline.main()
    rep = json.loads((OUTPUTS / "reports" / "baseline_logreg_classification_report.json").read_text())
    summary["baseline_macro_f1"] = round(rep["macro avg"]["f1-score"], 3)
    if cnn:
        t = step("3b", f"Deep model: {cnn}")
        try:
            import train_cnn; train_cnn.main(cnn, epochs=8, lr=3e-4, bs=32, patience=3, no_noisy_copies=False)
        except Exception as e:      # keep the demo going even if the deep model fails
            print(f"!! Deep model step failed: {e!r} -- continuing with the baseline model")
    t = step(5, "7-day whale presence forecast")
    import forecast; summary["forecast"] = {k: {m: round(v, 3) for m, v in d.items()} for k, d in forecast.main().items()}
    t = step(6, "AIS ship traffic + dynamic ship-strike risk + 10-knot what-if")
    import risk_engine; summary["risk"] = risk_engine.main()
    t = step(7, "Plain-language advisories")
    import advisory; summary["example_advisory"] = advisory.main()[0]
    summary["runtime_s"] = round(time.time() - t_all, 1)
    (OUTPUTS / "reports").mkdir(parents=True, exist_ok=True)
    (OUTPUTS / "reports" / "pipeline_summary.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False))
    print(f"\nDONE in {summary['runtime_s']} s. Outputs in {OUTPUTS}")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--sample", action="store_true", help="generate and use synthetic sample data")
    ap.add_argument("--cnn", choices=["custom_cnn", "resnet50", "efficientnet_b0"], help="also train a deep model")
    ap.add_argument("--clips-per-class", type=int, default=60)
    a = ap.parse_args()
    main(a.sample, a.cnn, a.clips_per_class)

Overwriting src/run_pipeline.py



In [ ]:
%%writefile src/download_data.py
"""Download the REAL data (run on your laptop; needs internet).

    python src/download_data.py sanctsound --sites CI01 CI02 CI03 CI04 CI05
    python src/download_data.py ais --start 2021-06-01 --days 14

Watkins clips and NOAA raw audio are downloaded by hand — see README section "Real data".
"""
import argparse
import io
import urllib.parse
import urllib.request
import zipfile

import pandas as pd

from config import DETECTIONS_CSV, AIS_DIR, STUDY_BBOX

ERDDAP = "https://coastwatch.pfeg.noaa.gov/erddap"
SPECIES_KEYS = {"bluewhale": "blue_whale", "finwhale": "fin_whale", "humpbackwhale": "humpback_whale"}


def fetch(url, timeout=120):
    req = urllib.request.Request(url, headers={"User-Agent": "whale-capstone/1.0"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.read()


def sanctsound(sites):
    """Find SanctSound daily-presence datasets on NOAA ERDDAP and convert them to
    date, site, species, present."""
    q = urllib.parse.quote("SanctSound")
    listing = pd.read_csv(io.BytesIO(fetch(f"{ERDDAP}/search/index.csv?page=1&itemsPerPage=2000&searchFor={q}")))
    ids = listing["Dataset ID"].astype(str)
    rows = []
    for ds in ids:
        low = ds.lower()
        site = next((s for s in sites if f"_{s.lower()}_" in low), None)
        sp = next((v for k, v in SPECIES_KEYS.items() if k in low), None)
        if not site or not sp or not (low.endswith("_1d") or "_1d_" in low):
            continue
        kind = "griddap" if ds in set(listing.loc[listing["griddap"].notna(), "Dataset ID"]) else "tabledap"
        try:
            df = pd.read_csv(io.BytesIO(fetch(f"{ERDDAP}/{kind}/{ds}.csv")), skiprows=[1])   # row 2 = units
        except Exception as e:
            print("  could not download", ds, e); continue
        tcol = next(c for c in df.columns if c.lower().startswith("time"))
        vcol = next(c for c in df.columns if c != tcol and pd.api.types.is_numeric_dtype(df[c])
                    and c.lower() not in ("latitude", "longitude", "depth"))
        d = pd.DataFrame({"date": pd.to_datetime(df[tcol]).dt.date.astype(str), "site": site, "species": sp,
                          "present": (df[vcol].fillna(0) > 0).astype(int)})
        rows.append(d)
        print(f"  {ds}: {len(d)} days ({vcol})")
    if not rows:
        raise SystemExit("No matching datasets found — download manually from the ERDDAP search page (README).")
    out = pd.concat(rows).groupby(["date", "site", "species"], as_index=False)["present"].max()
    DETECTIONS_CSV.parent.mkdir(parents=True, exist_ok=True)
    out.to_csv(DETECTIONS_CSV, index=False)
    print("saved", DETECTIONS_CSV, len(out), "rows. Now fill data/detections/sites.csv with site lat/lon (from ERDDAP metadata).")


def ais(start, days):
    """MarineCadastre daily national files (~hundreds of MB each). We keep only the study area,
    so each day shrinks to a few MB."""
    AIS_DIR.mkdir(parents=True, exist_ok=True)
    b = STUDY_BBOX
    for d in pd.date_range(start, periods=days, freq="D"):
        url = f"https://coast.noaa.gov/htdata/CMSP/AISDataHandler/{d.year}/AIS_{d:%Y_%m_%d}.zip"
        out = AIS_DIR / f"ais_socal_{d:%Y_%m_%d}.csv"
        if out.exists():
            continue
        print("downloading", url)
        try:
            z = zipfile.ZipFile(io.BytesIO(fetch(url, timeout=600)))
        except Exception as e:
            print("  failed:", e, "(check the year's file format on marinecadastre.gov/ais)"); continue
        keep = []
        with z.open(z.namelist()[0]) as f:
            for chunk in pd.read_csv(f, chunksize=500_000):
                chunk.columns = [c.strip() for c in chunk.columns]
                lat = "LAT" if "LAT" in chunk else "latitude"
                lon = "LON" if "LON" in chunk else "longitude"
                keep.append(chunk[chunk[lat].between(b["lat_min"], b["lat_max"]) & chunk[lon].between(b["lon_min"], b["lon_max"])])
        pd.concat(keep).to_csv(out, index=False)
        print("  saved", out)


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    sub = ap.add_subparsers(dest="cmd", required=True)
    s = sub.add_parser("sanctsound"); s.add_argument("--sites", nargs="+", default=["CI01", "CI02", "CI03", "CI04", "CI05"])
    a = sub.add_parser("ais"); a.add_argument("--start", default="2021-06-01"); a.add_argument("--days", type=int, default=7)
    args = ap.parse_args()
    sanctsound(args.sites) if args.cmd == "sanctsound" else ais(args.start, args.days)

Writing src/download_data.py



keyboard_arrow_down
1–6 · Run the complete flow (sample data, baseline model, forecast, risk, advisories)


In [ ]:
# 1. Programmatically write out make_sample_data.py to ensure it exists
import os
os.makedirs('/content/whale_ai/src', exist_ok=True)

with open('/content/whale_ai/src/make_sample_data.py', 'w') as f:
    f.write('''"""Synthetic sample data generator."""
import argparse
import numpy as np
import pandas as pd
from scipy.io import wavfile
from config import RAW_CLIPS, NOISE_DIR, DETECTIONS_CSV, SITES_CSV, AIS_DIR, SAMPLE_RATE as SR

rng = np.random.default_rng(7)

def chirp(f0, f1, dur, sr=SR, harmonics=1, curve=1.0):
    t = np.arange(int(dur * sr)) / sr
    frac = (t / dur) ** curve
    f = f0 + (f1 - f0) * frac
    phase = 2 * np.pi * np.cumsum(f) / sr
    y = sum(np.sin(h * phase) / h for h in range(1, harmonics + 1))
    env = np.hanning(len(t))
    return y * env

def pink(n):
    w = rng.standard_normal(n)
    X = np.fft.rfft(w)
    X /= np.sqrt(np.arange(1, len(X) + 1))
    return np.fft.irfft(X, n)

def ship_noise(n, sr=SR):
    t = np.arange(n) / sr
    brown = np.cumsum(rng.standard_normal(n)); brown -= np.convolve(brown, np.ones(2001) / 2001, mode="same")
    blade = rng.uniform(6, 12)
    tonals = sum(np.sin(2 * np.pi * blade * k * t + rng.uniform(0, 6)) / k for k in range(3, 30))
    am = 1 + 0.5 * np.sin(2 * np.pi * rng.uniform(1, 3) * t)
    y = 0.6 * brown / (np.std(brown) + 1e-9) + 0.4 * tonals / (np.std(tonals) + 1e-9)
    return y * am

def place(canvas, call, start):
    end = min(len(canvas), start + len(call))
    canvas[start:end] += call[:end - start]

def make_call(species, n):
    y = np.zeros(n)
    j = lambda a, b: rng.uniform(a, b)
    if species == "blue_whale":
        place(y, chirp(j(16, 19), j(15, 17), j(2.0, 3.0)), int(j(0, 0.8) * SR))
        place(y, chirp(j(48, 52), j(38, 42), j(1.2, 1.8)), int(j(2.5, 3.2) * SR))
    elif species == "fin_whale":
        for k in range(3):
            place(y, chirp(j(26, 30), j(17, 19), j(0.8, 1.1)), int((0.3 + k * j(1.4, 1.7)) * SR))
    elif species == "humpback_whale":
        t0 = j(0, 0.5)
        for _ in range(int(j(2, 4))):
            d = j(0.5, 1.2)
            place(y, chirp(j(200, 600), j(400, 1500), d, harmonics=4, curve=j(0.5, 2)), int(t0 * SR))
            t0 += d + j(0.2, 0.5)
    elif species == "killer_whale":
        t0 = j(0, 1)
        for _ in range(int(j(2, 4))):
            d = j(0.4, 0.9)
            place(y, chirp(j(900, 1400), j(1500, 2500), d, harmonics=3, curve=0.3), int(t0 * SR))
            t0 += d + j(0.3, 0.6)
    elif species == "bottlenose_dolphin":
        t0 = j(0, 1)
        for _ in range(int(j(2, 4))):
            d = j(0.3, 0.8)
            f0 = j(4000, 6000)
            t = np.arange(int(d * SR)) / SR
            f = f0 + j(500, 1500) * np.sin(2 * np.pi * j(1, 3) * t)
            place(y, np.sin(2 * np.pi * np.cumsum(f) / SR) * np.hanning(len(t)), int(t0 * SR))
            t0 += d + j(0.2, 0.5)
        for c in np.arange(j(0, 1), 4.5, j(0.05, 0.12)):
            i = int(c * SR); y[i:i + 8] += rng.standard_normal(min(8, n - i)) * 0.8
    return y

def make_clip(species, dur=5.0):
    n = int(dur * SR)
    ambient = pink(n); ambient /= np.std(ambient)
    if species == "background":
        y = ship_noise(n) + 0.3 * ambient
    else:
        call = make_call(species, n); call /= np.std(call) + 1e-9
        snr = rng.uniform(3, 20)
        y = call + ambient * 10 ** (-snr / 20)
    return (y / (np.max(np.abs(y)) + 1e-9) * 0.9).astype(np.float32)

SPECIES = ["blue_whale", "fin_whale", "humpback_whale", "killer_whale", "bottlenose_dolphin", "background"]
SITES = pd.DataFrame({"site": ["CI01", "CI02", "CI03", "CI04", "CI05"],
                      "lat": [34.05, 33.48, 33.90, 34.28, 33.93],
                      "lon": [-120.03, -119.05, -119.60, -120.70, -118.95]})
PEAK_DOY = {"blue_whale": 240, "fin_whale": 330, "humpback_whale": 180}

def make_detections():
    rows = []
    days = pd.date_range("2019-01-01", "2022-12-31", freq="D")
    for site in SITES.site:
        base = rng.uniform(0.6, 1.2)
        for sp, peak in PEAK_DOY.items():
            state = 0
            for d in days:
                season = 0.5 * (1 + np.cos(2 * np.pi * (d.dayofyear - peak) / 365))
                p_on = np.clip(0.02 + 0.35 * season * base, 0, 0.9)
                p_stay = np.clip(0.55 + 0.4 * season, 0, 0.97)
                state = int(rng.random() < (p_stay if state else p_on))
                rows.append((d.date().isoformat(), site, sp, state))
    return pd.DataFrame(rows, columns=["date", "site", "species", "present"])

def make_ais(n_vessels=160):
    lanes = [np.array([[34.45, -120.80], [34.25, -119.90], [34.05, -119.20], [33.75, -118.30]]),
             np.array([[33.60, -121.20], [33.45, -119.80], [33.55, -118.70], [33.72, -118.25]])]
    rows = []
    for v in range(n_vessels):
        lane = lanes[rng.integers(0, 2)]
        if rng.random() < 0.5:
            lane = lane[::-1]
        start = pd.Timestamp("2022-06-01") + pd.Timedelta(hours=float(rng.uniform(0, 24 * 150)))
        sog = float(np.clip(rng.normal(13.5, 2.5), 6, 20))
        vtype = int(rng.choice([70, 71, 80, 60, 30, 37], p=[.35, .1, .25, .1, .1, .1]))
        pts = []
        for a, b in zip(lane[:-1], lane[1:]):
            for f in np.linspace(0, 1, 40, endpoint=False):
                pts.append(a + f * (b - a) + rng.normal(0, 0.02, 2))
        km_per_step = 230 / len(pts)
        t = start
        for p in pts:
            speed = max(3, sog + rng.normal(0, 0.8))
            rows.append((100000000 + v, t.strftime("%Y-%m-%dT%H:%M:%S"), round(p[0], 5), round(p[1], 5),
                         round(speed, 1), vtype))
            t += pd.Timedelta(hours=km_per_step / (speed * 1.852))
    return pd.DataFrame(rows, columns=["MMSI", "BaseDateTime", "LAT", "LON", "SOG", "VesselType"])

def main(clips_per_class=60):
    for sp in SPECIES:
        d = RAW_CLIPS / sp
        d.mkdir(parents=True, exist_ok=True)
        for i in range(clips_per_class):
            wavfile.write(d / f"synthetic_{sp}_{i:03d}.wav", SR, make_clip(sp, dur=rng.uniform(5, 9)))
    NOISE_DIR.mkdir(parents=True, exist_ok=True)
    for i in range(10):
        n = ship_noise(SR * 20); wavfile.write(NOISE_DIR / f"synthetic_ship_noise_{i}.wav", SR,
                                               (n / np.abs(n).max() * 0.9).astype(np.float32))
    DETECTIONS_CSV.parent.mkdir(parents=True, exist_ok=True)
    make_detections().to_csv(DETECTIONS_CSV, index=False)
    SITES.to_csv(SITES_CSV, index=False)
    AIS_DIR.mkdir(parents=True, exist_ok=True)
    make_ais().to_csv(AIS_DIR / "sample_ais_2022.csv", index=False)
    print(f"Synthetic sample data written under {RAW_CLIPS.parent}")

if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--clips-per-class", type=int, default=60)
    main(ap.parse_args().clips_per_class)
''')

# 2. Run the pipeline with PYTHONPATH properly specified
!cd /content/whale_ai/src && PYTHONPATH=. python3 run_pipeline.py --sample


======================================================================
STEP 1: Create synthetic sample data (replace with real data later)
======================================================================
Synthetic sample data written under /content/whale_ai/data

======================================================================
STEP 2: Audio -> 5 s windows -> log-mel spectrograms (Bronze -> Silver)
======================================================================
  100/360 files
  200/360 files
  300/360 files
Saved (766, 128, 310) spectrograms to /content/whale_ai/data/silver
split               test  train  val
species                             
background            12     56   15
blue_whale            12    118   14
bottlenose_dolphin    15    106   11
fin_whale             13    114   11
humpback_whale        13    112   11
killer_whale          12    112    9

======================================================================
STEP 3: Species classifier: classical baseline (+ noise robustness test)
======================================================================
[tracking] baseline_logreg: {"accuracy": 0.9481, "macro_f1": 0.9466, "macro_auc": 0.9983, "f1_snr_clean": 0.9359, "f1_snr_20": 0.9517, "f1_snr_10": 0.952, "f1_snr_5": 0.9679, "f1_snr_0": 0.7877}

======================================================================
STEP 5: 7-day whale presence forecast
======================================================================
[tracking] forecast_seasonal_baseline: {"auc": 0.8326, "brier": 0.1513, "f1": 0.8614}
[tracking] forecast_logistic_regression: {"auc": 0.8715, "brier": 0.1281, "f1": 0.8697}
[tracking] forecast_gradient_boosting: {"auc": 0.8765, "brier": 0.1261, "f1": 0.8822}
                       auc  brier     f1
seasonal_baseline    0.833  0.151  0.861
logistic_regression  0.871  0.128  0.870
gradient_boosting    0.876  0.126  0.882

======================================================================
STEP 6: AIS ship traffic + dynamic ship-strike risk + 10-knot what-if
======================================================================
/usr/local/lib/python3.13/dist-packages/folium/raster_layers.py:130: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tiles = tiles.build_url(fill_subdomain=False, scale_factor="{r}")  # type: ignore
{'cells': 248, 'high_cells': 25, 'risk_reduction_if_10kn_pct': np.float64(51.7), 'high_cell_reduction_pct': np.float64(51.6), 'ais_points': 16200, 'vessels': 135, 'grid': 'h3'}

======================================================================
STEP 7: Plain-language advisories
======================================================================
[HIGH] Zone 862912d97ffffff (≈33.52°N, 119.03°W): whale presence likely this week (P≈0.89). Ships here average 14.0 kn. Reduce speed to 10 kn or less — this cuts lethal-strike risk by ~52%. Keep a lookout and report sightings.
[HIGH] Zone 862912d87ffffff (≈33.50°N, 119.10°W): whale presence likely this week (P≈0.87). Ships here average 13.8 kn. Reduce speed to 10 kn or less — this cuts lethal-strike risk by ~53%. Keep a lookout and report sightings.

DONE in 46.8 s. Outputs in /content/whale_ai/outputs



keyboard_arrow_down
2 · Deep models on the GPU — transfer learning benchmark

Every run is tracked (MLflow if installed, otherwise outputs/runs/).


In [ ]:
!python src/train_cnn.py --model custom_cnn --epochs 25 --patience 6
!python src/train_cnn.py --model resnet50 --epochs 12 --patience 4
!python src/train_cnn.py --model efficientnet_b0 --epochs 12 --patience 4

Traceback (most recent call last):
  File "/content/whale_ai/src/train_cnn.py", line 163, in <module>
    main(a.model, a.epochs, a.lr, a.batch_size, a.patience, a.no_noisy_copies)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/whale_ai/src/train_cnn.py", line 104, in main
    X, y, meta, classes = load_silver(include_noisy_copies=not no_noisy_copies)
                          ~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/whale_ai/src/evaluate.py", line 15, in load_silver
    X = np.load(SILVER / "specs.npy").astype(np.float32)
        ~~~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/numpy/lib/_npyio_impl.py", line 459, in load
    fid = stack.enter_context(open(os.fspath(file), "rb"))
                              ~~~~^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/content/whale_ai/data/silver/specs.npy'
Traceback (most recent call last):
  File "/content/whale_ai/src/train_cnn.py", line 163, in <module>
    main(a.model, a.epochs, a.lr, a.batch_size, a.patience, a.no_noisy_copies)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/whale_ai/src/train_cnn.py", line 104, in main
    X, y, meta, classes = load_silver(include_noisy_copies=not no_noisy_copies)
                          ~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/whale_ai/src/evaluate.py", line 15, in load_silver
    X = np.load(SILVER / "specs.npy").astype(np.float32)
        ~~~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/numpy/lib/_npyio_impl.py", line 459, in load
    fid = stack.enter_context(open(os.fspath(file), "rb"))
                              ~~~~^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/content/whale_ai/data/silver/specs.npy'
Traceback (most recent call last):
  File "/content/whale_ai/src/train_cnn.py", line 163, in <module>
    main(a.model, a.epochs, a.lr, a.batch_size, a.patience, a.no_noisy_copies)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/whale_ai/src/train_cnn.py", line 104, in main
    X, y, meta, classes = load_silver(include_noisy_copies=not no_noisy_copies)
                          ~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/whale_ai/src/evaluate.py", line 15, in load_silver
    X = np.load(SILVER / "specs.npy").astype(np.float32)
        ~~~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/numpy/lib/_npyio_impl.py", line 459, in load
    fid = stack.enter_context(open(os.fspath(file), "rb"))
                              ~~~~^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/content/whale_ai/data/silver/specs.npy'



keyboard_arrow_down
Model comparison table


In [ ]:
import json, glob, pandas as pd
rows = []
for f in sorted(glob.glob("outputs/runs/*/run.json")):
    r = json.load(open(f)); m = r["final_metrics"]
    if r["tags"].get("module") != "A": continue
    rows.append({"model": r["name"],
                 "test macro-F1": m.get("test_macro_f1", m.get("macro_f1")),
                 "test accuracy": m.get("test_accuracy", m.get("accuracy")),
                 "macro AUC": m.get("test_macro_auc", m.get("macro_auc")),
                 "F1 @ 5 dB noise": m.get("f1_snr_5"), "F1 @ 0 dB noise": m.get("f1_snr_0")})

if len(rows) > 0:
    comparison = pd.DataFrame(rows).drop_duplicates("model", keep="last").sort_values("test macro-F1", ascending=False)
    display(comparison.round(3).reset_index(drop=True))
else:
    print("No model runs found under outputs/runs/. Please run the pipeline or training cells first.")

No model runs found under outputs/runs/. Please run the pipeline or training cells first.



keyboard_arrow_down
3 · Figures: spectrograms, confusion matrices, noise curves, Grad-CAM


In [ ]:
from IPython.display import Image, display, Markdown
figs = ["spectrogram_gallery.png", "baseline_logreg_confusion.png", "custom_cnn_confusion.png",
        "resnet50_confusion.png", "efficientnet_b0_confusion.png",
        "baseline_logreg_noise_curve.png", "custom_cnn_noise_curve.png", "resnet50_noise_curve.png",
        "custom_cnn_gradcam.png", "resnet50_gradcam.png", "efficientnet_b0_gradcam.png"]
for f in figs:
    p = f"outputs/figures/{f}"
    if os.path.exists(p):
        display(Markdown(f"**{f}**")); display(Image(p, width=900 if "gallery" in f or "gradcam" in f else 520))



keyboard_arrow_down
4 · 7-day whale-presence forecast


In [ ]:
# The FileNotFoundError occurs because the pipeline in cell xAoMNo-9X0xp hasn't run successfully yet.
# Since 'run_pipeline.py' failed on step 1, the 'outputs/' directory and its files do not exist.
# Once the pipeline completes successfully, this cell will execute correctly.

import os
import pandas as pd
from IPython.display import Image, display

metrics_path = "outputs/tables/forecast_metrics.csv"
image_path = "outputs/figures/forecast_example.png"

if os.path.exists(metrics_path):
    display(pd.read_csv(metrics_path, index_col=0))
else:
    print(f"[!] {metrics_path} not found. Please run the pipeline cell (xAoMNo-9X0xp) successfully first.")

if os.path.exists(image_path):
    display(Image(image_path, width=900))
else:
    print(f"[!] {image_path} not found.")

[!] outputs/tables/forecast_metrics.csv not found. Please run the pipeline cell (xAoMNo-9X0xp) successfully first.
[!] outputs/figures/forecast_example.png not found.



keyboard_arrow_down
5 · Ship-strike risk map (click a cell for details) + 10-knot what-if


In [ ]:
# This FileNotFoundError occurs because the pipeline in cell xAoMNo-9X0xp has not run successfully yet.
# Once the pipeline finishes, the risk table and maps will be generated.

import os
import pandas as pd
from IPython.display import Image, display, HTML

risk_path = "outputs/tables/risk_cells.csv"
map_html = "outputs/figures/risk_map.html"
map_png = "outputs/figures/risk_map.png"

if os.path.exists(risk_path):
    risk = pd.read_csv(risk_path)
    print(f"High-risk cells: {(risk.risk_level == 'HIGH').sum()}  |  "
          f"risk cut if all ships slow to 10 kn: {100 * (1 - risk.risk_if_10kn.sum() / risk.risk.sum()):.0f}%")

    if os.path.exists(map_html):
        display(HTML(open(map_html).read()))
    elif os.path.exists(map_png):
        display(Image(map_png))
else:
    print(f"[!] {risk_path} not found. Please run the pipeline cell (xAoMNo-9X0xp) successfully first.")

[!] outputs/tables/risk_cells.csv not found. Please run the pipeline cell (xAoMNo-9X0xp) successfully first.



keyboard_arrow_down
6 · Advisories for ships


In [ ]:
print(open("outputs/reports/advisories.txt").read())



keyboard_arrow_down
7 · Live demo — identify the species in a recording


In [ ]:
import numpy as np, matplotlib.pyplot as plt, torch, IPython.display as ipd
import audio_utils as au
from train_cnn import build_model, device, predict, last_conv_layer
from gradcam import gradcam

best = comparison[comparison.model.isin(["custom_cnn", "resnet50", "efficientnet_b0"])].iloc[0]["model"]
ck = torch.load(f"outputs/models/{best}.pt", map_location="cpu")
dev = device(); model = build_model(ck["name"], len(ck["classes"])); model.load_state_dict(ck["state"]); model.to(dev).eval()
classes = ck["classes"]; print("Using best model:", best)

def identify(path):
    wins = au.segment(au.normalise(au.bandpass(au.load_audio(path))))
    specs = [au.log_mel(w) for w in wins]
    proba = predict(model, specs, dev); mean_p = proba.mean(0); k = int(mean_p.argmax())
    i = int(proba[:, k].argmax())
    cam, _ = gradcam(model, specs[i], k, last_conv_layer(model), dev)
    ipd.display(ipd.Audio(au.load_audio(path), rate=au.SAMPLE_RATE))
    fig, ax = plt.subplots(1, 3, figsize=(15, 3.3), gridspec_kw={"width_ratios": [1.1, 1.4, 1.4]})
    ax[0].barh([c.replace("_", " ") for c in classes], mean_p, color="#8E0A13"); ax[0].set_xlim(0, 1)
    ax[0].set_title(f"Detected: {classes[k].replace('_', ' ')} ({mean_p[k]:.0%})")
    ax[1].imshow(specs[i], origin="lower", aspect="auto", cmap="magma"); ax[1].set_title("log-mel spectrogram")
    ax[2].imshow(specs[i], origin="lower", aspect="auto", cmap="gray")
    ax[2].imshow(cam, origin="lower", aspect="auto", cmap="jet", alpha=.45); ax[2].set_title("Grad-CAM: where the model looked")
    for a in ax[1:]: a.set_xticks([]); a.set_yticks([])
    plt.tight_layout(); plt.show()

import glob, ipywidgets as w
demo = {os.path.basename(f): f for f in sorted(glob.glob("data/raw_clips/*/*.wav"))[::12]}
w.interact(lambda recording: identify(demo[recording]), recording=w.Dropdown(options=list(demo)));



In [ ]:
keyboard_arrow_down
Or upload your own WAV/FLAC recording
#@title Or upload your own WAV/FLAC recording
from google.colab import files
for name, data in files.upload().items():
    open(name, "wb").write(data); identify(name)



keyboard_arrow_down
Optional · Log runs to Databricks MLflow (reviewer recommendation)

Create a token in Databricks (User settings → Developer → Access tokens), fill in below, run, then re-run the training cell. Runs appear under Experiments → /Shared/whale-dolphin-capstone.


In [ ]:
# !pip install -q mlflow
# import os
# os.environ["DATABRICKS_HOST"] = "https://<your-workspace>.cloud.databricks.com"
# os.environ["DATABRICKS_TOKEN"] = "<paste token - do not share the notebook with it>"
# os.environ["MLFLOW_TRACKING_URI"] = "databricks"

